# Multi-Draft Diversity on Colab

This notebook clones the repo, checks out `feature/multidraft`, installs the minimal dependencies needed for the new `multidraft` strategy, and runs a small prompt-only pilot.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Edit these paths if needed.
REPO_URL = "https://github.com/Peregalli/llm-reasoning-mt-EXT.git"
BRANCH = "feature/multidraft"

# Input data: this folder must contain Xhosa.jsonl
INPUT_DIR = "/content/drive/MyDrive/NLP_data"

# Output data will be written here
OUTPUT_DIR = "/content/drive/MyDrive/NLP_outputs/multidraft_qwen15b"

# Model and experiment settings
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
NUM_SAMPLES = 20
REQUEST_BATCH_SIZE = 1
MAX_NEW_TOKENS = 256
DRAFT_STYLES = ["faithful", "fluent", "balanced"]

# Optional: if you need gated model access, set a HF token here.
HF_TOKEN = ""


In [ ]:
%cd /content
!rm -rf llm-reasoning-mt-EXT
!git clone {REPO_URL}
%cd /content/llm-reasoning-mt-EXT
!git checkout {BRANCH}


In [ ]:
!pip install -q torch transformers datasets accelerate peft trl huggingface-hub safetensors sentencepiece protobuf numpy tqdm scipy scikit-learn sacrebleu termcolor openai anthropic cohere


In [ ]:
import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(HF_TOKEN)
else:
    print("HF_TOKEN is empty. This is fine for public models.")


In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

styles = " ".join(DRAFT_STYLES)
cmd = f"""
python paraphrase.py \
  --strategy multidraft \
  --model_name_or_path {MODEL_NAME} \
  --tokenizer_name_or_path {MODEL_NAME} \
  --inference_api hf \
  --request_batch_size {REQUEST_BATCH_SIZE} \
  --seed 122 \
  --max_new_tokens {MAX_NEW_TOKENS} \
  --temperature 0.3 \
  --top_p 0.95 \
  --repetition_penalty 1.0 \
  --num_return_sequences 1 \
  --num_beams 1 \
  --do_sample \
  --verbose \
  --languages Xhosa \
  --input_filenames Xhosa.jsonl \
  --input_dir {INPUT_DIR} \
  --output_dir {OUTPUT_DIR} \
  --source_language English \
  --number_of_generations_per_step 3 \
  --draft_styles {styles} \
  --max_samples {NUM_SAMPLES}
"""
print(cmd)
!{cmd}


In [ ]:
!ls -lah {OUTPUT_DIR}
!head -n 3 {OUTPUT_DIR}/Xhosa_paraphrase_multidraft.jsonl
